# 09주차 · 임베딩과 딥러닝 기반 NLP
## Word2Vec · FastText · 문장 벡터 · RNN
**충북대학교 | 자연언어처리 | 임화정**

### 학습목표
- 희소 표현과 분산 표현의 차이, 분포 가설을 설명한다.
- CBOW/Skip-gram의 입력·예측 방향을 구분한다.
- gensim으로 Word2Vec/FastText를 학습하고 코사인 유사도를 계산한다.
- 학습 어휘와 OOV 벡터 생성의 차이를 설명한다.
- 평균 문장 벡터의 어순 손실과 RNN의 순차 처리를 비교한다.

### 실행 안내
Colab/Jupyter에서 위에서부터 실행합니다. CPU로 충분하며 GPU·JVM·사전학습 대형 모델은 필요 없습니다. Python 3.10 이상, numpy, pandas, gensim을 사용합니다. Colab에서 패키지가 없다면 아래 명령을 별도 코드 셀로 실행하세요. 설치에는 인터넷이 필요하며, 설치 후에도 import가 실패하면 런타임을 재시작하고 처음부터 실행하세요.
```python
%pip install -U gensim numpy scipy pandas
```

**데이터와 결과:** 말뭉치는 자체 작성한 교육용 템플릿 80문장입니다. 실제 뉴스/리뷰 데이터가 아니며, 작은 말뭉치의 유사어 순위를 언어 의미의 정답으로 볼 수 없습니다. 반복 학습은 고유 데이터의 다양성을 늘려주지 않습니다.

**학생용: TODO A/B/C/E·OOV 과제·응답란 포함**

> **[제출 안내]** 작성자: 송용휘
>
> - **바꾼 원본 셀:** ① TODO 줄(실습 A·B·C·E), ② 설치 셀(`!pip install gensim` → `%pip install -q gensim`, 설치 출력만 간소화). 나머지 원본 셀은 문구 그대로입니다.
> - **추가한 셀:** 코드 셀은 첫 줄 주석, 마크다운 셀은 맨 앞에 **[과제] · [선택 확장] · [추가 검증] · [해석 질문 근거] · [해석] · [답변] · [제출 기록] · [추가] · [제출 안내]** 중 하나를 표시했습니다.
> - **실행:** 커널을 새로 시작해 위에서부터 한 번에 실행한 결과가 저장되어 있습니다.
> - **데이터:** 교육용으로 자체 작성한 템플릿 80문장이며 실제 뉴스·리뷰 데이터가 아닙니다. 아래 유사어 순위와 지표는 이 말뭉치의 구조를 진단한 값이지 언어 의미의 정답이나 분류 성능이 아닙니다.

In [1]:
# Colab에 gensim이 없을 때 설치 (설치 후에도 import가 실패하면 런타임 재시작 후 처음부터 실행)
%pip install -q gensim

Note: you may need to restart the kernel to use updated packages.


## 0. 환경·재현성
`workers=1`과 시드를 고정합니다. Python 해시, 라이브러리 버전, 연산 구현에 따라 런타임 간 결과가 달라질 수 있습니다. 엄격한 재현 실험은 프로세스 시작 전 PYTHONHASHSEED도 고정해야 합니다. Colab에서는 같은 실행 환경의 조건을 기록하는 것으로 시작하세요.

In [2]:
import sys
import numpy as np
import pandas as pd
import gensim
from gensim.models import Word2Vec, FastText
from IPython.display import display
SEED = 42
print('Python:', sys.version.split()[0])
print('numpy:', np.__version__, '| pandas:', pd.__version__, '| gensim:', gensim.__version__)


Python: 3.13.18
numpy: 2.5.4 | pandas: 3.1.1 | gensim: 4.4.0


## 1. 분포 가설과 교육용 말뭉치
비슷한 문맥에 등장하는 단어가 비슷한 표현을 얻는다는 가설입니다. 같은 문맥에 반의어도 등장할 수 있으므로 **유사도=동의어 여부**가 아닙니다.

아래 문장은 이미 공백 단위로 구분된 토큰 예시입니다. 일반 한국어 원문에 공백 분할을 적용한 것과 다르며 형태소 분석 결과라고 주장하지 않습니다. 분야마다 4단어 × 5문맥 = 20문장, 전체 80문장입니다. 분야끼리 문맥이 인위적으로 분리되어 있다는 편향을 기록하세요.

In [3]:
groups = {
    '동물': (['강아지', '개', '고양이', '토끼'], [
        '나는 {w} 를 좋아한다', '{w} 는 귀여운 동물 이다',
        '집 에서 {w} 와 함께 지낸다', '{w} 에게 먹이 를 준다',
        '공원 에서 {w} 와 산책 한다']),
    '음식': (['사과', '바나나', '딸기', '포도'], [
        '나는 {w} 를 먹는다', '{w} 는 맛있는 과일 이다',
        '시장 에서 신선한 {w} 를 산다', '{w} 로 주스 를 만든다',
        '아침 에 달콤한 {w} 를 먹는다']),
    '영화': (['영화', '드라마', '공연', '연극'], [
        '나는 {w} 를 감상 한다', '{w} 의 배우 연기 가 훌륭하다',
        '{w} 의 이야기 가 재미있다', '친구 와 {w} 를 보고 감동 했다',
        '{w} 의 결말 과 연출 이 좋다']),
    '제품': (['휴대폰', '노트북', '태블릿', '컴퓨터'], [
        '나는 {w} 를 사용 한다', '{w} 의 화면 이 밝다',
        '{w} 의 배터리 가 오래 간다', '새로운 {w} 의 성능 이 좋다',
        '{w} 의 가격 과 품질 을 비교 한다'])
}
records = []
for domain, (words, templates) in groups.items():
    for word in words:
        for template in templates:
            records.append({'domain': domain, 'text': template.format(w=word)})
corpus_df = pd.DataFrame(records)
sentences = [text.split() for text in corpus_df.text]
assert len(sentences) == 80
assert not corpus_df.text.duplicated().any()
vocabulary = sorted({word for sent in sentences for word in sent})
print('문장:', len(sentences), '| 고유 토큰:', len(vocabulary))
display(corpus_df.groupby('domain').size().to_frame('sentences'))
print('첫 문장 토큰:', sentences[0])


문장: 80 | 고유 토큰: 75


,sentences
domain,
동물,20
영화,20
음식,20
제품,20


첫 문장 토큰: ['나는', '강아지', '를', '좋아한다']


## 2. 중심 단어와 문맥 쌍
Skip-gram은 중심 → 문맥, CBOW는 주변 문맥 → 중심을 예측합니다. `window=2`이면 경계에서는 주변 단어 수가 줄어듭니다. 아래는 **고정 윈도우 개념 예시**이며 gensim 기본 학습은 윈도우를 확률적으로 줄일 수 있습니다.

### 실습 A
중심 위치를 제외하고 최대 window 거리의 단어를 수집하는 함수를 완성하세요.

In [4]:
def context_words(tokens, center_index, window=2):
    # TODO A (완료): 좌우 범위를 계산하고 중심 위치를 제외하세요.
    if window < 0:
        raise ValueError('window는 0 이상이어야 합니다.')
    if not 0 <= center_index < len(tokens):
        raise IndexError('center_index가 문장 범위를 벗어났습니다.')
    start = max(0, center_index - window)               # 왼쪽 끝: 문장 시작(0)보다 앞으로 가지 않음
    end = min(len(tokens), center_index + window + 1)   # 오른쪽 끝(미포함): 문장 길이를 넘지 않음
    return [tokens[i] for i in range(start, end) if i != center_index]  # 중심 위치만 제외

example = ['나는', '귀여운', '강아지', '와', '산책', '한다']
contexts = context_words(example, 2, window=2)
print('CBOW:', contexts, '→', example[2])
print('Skip-gram:', [(example[2], c) for c in contexts])
print('완료 확인:', contexts == ['나는', '귀여운', '와', '산책'])


CBOW: ['나는', '귀여운', '와', '산책'] → 강아지
Skip-gram: [('강아지', '나는'), ('강아지', '귀여운'), ('강아지', '와'), ('강아지', '산책')]
완료 확인: True


In [5]:
# [추가 검증] 실습 A — 문장 경계에서는 문맥 수가 2×window보다 줄어들고, 중심 위치만 정확히 빠지는지 확인
cases = [  # (설명, center_index, window, 기대값)
    ('맨 앞 단어, window=2', 0, 2, ['귀여운', '강아지']),
    ('맨 끝 단어, window=2', 5, 2, ['와', '산책']),
    ('가운데, window=1', 2, 1, ['귀여운', '와']),
    ('가운데, window=0', 2, 0, []),
    ('문장보다 큰 window=10', 2, 10, ['나는', '귀여운', '와', '산책', '한다']),
]
for desc, idx, win, expected in cases:
    got = context_words(example, idx, window=win)
    assert got == expected, (desc, got, expected)
    print(f'{desc}: 중심={example[idx]} → 문맥 {len(got)}개 {got}')
print('경계 검증 통과')

맨 앞 단어, window=2: 중심=나는 → 문맥 2개 ['귀여운', '강아지']
맨 끝 단어, window=2: 중심=한다 → 문맥 2개 ['와', '산책']
가운데, window=1: 중심=강아지 → 문맥 2개 ['귀여운', '와']
가운데, window=0: 중심=강아지 → 문맥 0개 []
문장보다 큰 window=10: 중심=강아지 → 문맥 5개 ['나는', '귀여운', '와', '산책', '한다']
경계 검증 통과


## 3. Word2Vec 학습
입력 임베딩과 출력 임베딩이라는 두 가중치 행렬을 학습합니다. `wv`는 일반적으로 사용하는 입력 단어 벡터입니다. negative sampling의 negative는 '부정 감성'이 아니라 학습용 대조 단어 쌍을 뜻합니다.

|설정|의미|
|---|---|
|vector_size|단어 벡터 차원|
|window|최대 문맥 거리|
|min_count|최소 토큰 빈도|
|sg|0=CBOW, 1=Skip-gram|
|negative|정답 쌍당 대조 단어 수|
|epochs|학습 반복 횟수|
|sample=0|이 작은 예제에서 빈번한 토큰 subsampling을 끔|

### 실습 B
CBOW와 동일 조건으로 Skip-gram 모델을 만들되 sg를 변경하세요. 학생용 기본값은 CBOW이므로 수정 전에는 Skip-gram이라고 해석하지 마세요.

In [6]:
COMMON = dict(vector_size=32, window=2, min_count=1, workers=1,
              seed=SEED, epochs=150, negative=5, sample=0)
cbow = Word2Vec(sentences=sentences, sg=0, **COMMON)
# TODO B (완료): sg를 수정해 Skip-gram으로 만드세요. → sg=1 (중심 단어로 주변 문맥 단어를 예측)
skipgram = Word2Vec(sentences=sentences, sg=1, **COMMON)
print('CBOW sg:', cbow.sg, '| 두 번째 모델 sg:', skipgram.sg)
print('벡터 배열:', skipgram.wv.vectors.shape)
print('강아지 벡터 앞 8차원:', skipgram.wv['강아지'][:8])
print('완료 확인:', skipgram.sg == 1)


CBOW sg: 0 | 두 번째 모델 sg: 1
벡터 배열: (75, 32)
강아지 벡터 앞 8차원: [ 0.07492563 -0.7286843   0.7271578   0.43329677 -0.64734733 -0.88214636
 -0.79961336 -0.10343657]
완료 확인: True


## 4. 코사인 유사도와 유사어 탐색
코사인 유사도는 내적/(두 벡터의 길이 곱)입니다. 1은 같은 방향, -1은 반대 방향, 0은 직교를 뜻합니다. 0이 모든 의미 관계의 부재를 증명하는 것은 아닙니다. 벡터 길이가 0이면 코사인은 정의되지 않습니다.

### 실습 C
직접 계산한 코사인을 gensim 결과와 비교하세요.

In [7]:
def cosine_similarity_manual(a, b):
    # TODO C (완료): 분모가 0이면 np.nan, 그렇지 않으면 코사인 공식을 반환하세요.
    a = np.asarray(a, dtype=np.float64)   # float32 반올림 오차를 줄이기 위해 float64로 계산
    b = np.asarray(b, dtype=np.float64)
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return np.nan                     # 길이 0 벡터가 있으면 방향이 없어 코사인 정의 불가
    return float(np.dot(a, b) / denominator)

manual = cosine_similarity_manual(skipgram.wv['강아지'], skipgram.wv['개'])
reference = float(skipgram.wv.similarity('강아지', '개'))
print('직접 계산:', manual, '| gensim:', reference)
print('완료 확인:', np.isclose(manual, reference, atol=1e-6))
rows = []
for name, model in [('CBOW', cbow), ('두 번째 모델', skipgram)]:
    for word, score in model.wv.most_similar('강아지', topn=5):
        rows.append({'model': name, 'neighbor': word, 'cosine': score})
display(pd.DataFrame(rows))


직접 계산: 0.9969009081528966 | gensim: 0.9969009160995483
완료 확인: True


,model,neighbor,cosine
0,CBOW,개,0.997907
1,CBOW,고양이,0.997396
2,CBOW,토끼,0.996463
3,CBOW,지낸다,0.718324
4,CBOW,시장,0.710648
5,두 번째 모델,고양이,0.999099
6,두 번째 모델,토끼,0.997582
7,두 번째 모델,개,0.996901
8,두 번째 모델,지낸다,0.763945
9,두 번째 모델,준다,0.671197


In [8]:
# [추가 검증] 실습 C — 기하학적 의미, 0벡터 처리, 전체 어휘 쌍에서 gensim과 일치하는지 확인
e1, e2 = np.array([1.0, 0.0]), np.array([0.0, 1.0])
print('같은 방향(e1, 3·e1):', cosine_similarity_manual(e1, 3 * e1))
print('반대 방향(e1, -e1):', cosine_similarity_manual(e1, -e1))
print('직교(e1, e2)      :', cosine_similarity_manual(e1, e2))
print('0벡터 포함        :', cosine_similarity_manual(np.zeros(2), e1))
assert np.isnan(cosine_similarity_manual(np.zeros(2), e1))

kv = skipgram.wv
words = kv.index_to_key
errors = [abs(cosine_similarity_manual(kv[w1], kv[w2]) - float(kv.similarity(w1, w2)))
          for i, w1 in enumerate(words) for w2 in words[i + 1:]]
print(f'Skip-gram 어휘 {len(words)}개의 모든 쌍 {len(errors)}개: gensim과의 최대 절대오차 {max(errors):.1e}')
assert max(errors) < 1e-6

# 희소 표현(one-hot) vs 분산 표현: 서로 다른 단어의 one-hot 벡터는 항상 직교(cos 0)라 유사성을 표현하지 못함
one_hot = np.eye(len(words))
i, j = kv.key_to_index['강아지'], kv.key_to_index['개']
print(f"one-hot({len(words)}차원, 희소) cos(강아지, 개) = {cosine_similarity_manual(one_hot[i], one_hot[j])}"
      f" | Skip-gram({kv.vector_size}차원, 분산) cos(강아지, 개) = {cosine_similarity_manual(kv['강아지'], kv['개']):.4f}")

같은 방향(e1, 3·e1): 1.0
반대 방향(e1, -e1): -1.0
직교(e1, e2)      : 0.0
0벡터 포함        : nan
Skip-gram 어휘 75개의 모든 쌍 2775개: gensim과의 최대 절대오차 9.6e-08
one-hot(75차원, 희소) cos(강아지, 개) = 0.0 | Skip-gram(32차원, 분산) cos(강아지, 개) = 0.9969


### [과제] 4-1. 두 단어의 CBOW vs Skip-gram 유사어 순위 비교
- **단어 선택 (결과를 보기 전에 결정):** 위에서 본 '강아지'(동물) 외의 두 분야에서 한 단어씩 — 음식 **'사과'**, 제품 **'노트북'**.
- **동일 조건:** `COMMON`(seed=42, window=2, epochs=150 등)은 같고 `sg`만 다릅니다.
- 두 단어만으로는 우연일 수 있어 **분야 대상어 16개(4분야 × 4단어) 전체 요약**과 **seed 0–9 반복**으로 순위가 얼마나 안정적인지도 확인합니다.
- **지표:** `같은분야@3` = 상위 3 이웃 중 같은 분야 대상어의 비율, `분리폭` = 같은 분야 평균 cos − 다른 분야 평균 cos.
- `분야`는 말뭉치를 **생성한 규칙**에서 온 라벨일 뿐 의미의 정답이 아닙니다.

In [9]:
# [과제] CBOW vs Skip-gram — '사과'(음식), '노트북'(제품)의 상위 5개 이웃 비교 (sg만 다르고 나머지는 COMMON 동일)
import itertools
from collections import Counter

word2domain = {w: d for d, (ws, _) in groups.items() for w in ws}   # 분야 대상어 16개 → 분야
TARGETS = list(word2domain)
COMPARE_WORDS = ['사과', '노트북']
TOPN = 5

def neighbor_table(word, models, topn=TOPN):
    """모델별 상위 이웃을 순위별로 나란히 정리 (분야 대상어가 아닌 단어는 분야 '-')."""
    table = {}
    for name, model in models.items():
        nbrs = model.wv.most_similar(word, topn=topn)
        table[(name, '이웃')] = [w for w, _ in nbrs]
        table[(name, '분야')] = [word2domain.get(w, '-') for w, _ in nbrs]
        table[(name, 'cos')] = [round(float(s), 3) for _, s in nbrs]
    return pd.DataFrame(table, index=pd.RangeIndex(1, topn + 1, name='순위'))

def group_cos(model, dom_a, dom_b):
    """두 분야 대상어 사이의 평균 코사인."""
    sims = [float(model.wv.similarity(a, b)) for a in groups[dom_a][0] for b in groups[dom_b][0] if a != b]
    return float(np.mean(sims))

def domain_summary(model):
    """분야 대상어 16개 기준 진단 지표 — 템플릿 구조를 얼마나 복원했는지 보는 값이며 의미 품질 평가가 아님.
    같은분야@3 = 상위 3 이웃 중 같은 분야 대상어 비율, 분리폭 = 같은 분야 평균 cos − 다른 분야 평균 cos"""
    kv = model.wv
    same, cross = [], []
    for a, b in itertools.combinations(TARGETS, 2):
        (same if word2domain[a] == word2domain[b] else cross).append(float(kv.similarity(a, b)))
    p_at3 = np.mean([np.mean([word2domain.get(n) == word2domain[w] for n, _ in kv.most_similar(w, topn=3)])
                     for w in TARGETS])
    return {'같은분야@3': p_at3, '같은분야 cos': np.mean(same), '다른분야 cos': np.mean(cross),
            '분리폭': np.mean(same) - np.mean(cross), '영화–제품 cos': group_cos(model, '영화', '제품')}

models = {'CBOW': cbow, 'Skip-gram': skipgram}
for word in COMPARE_WORDS:
    print(f'■ {word} ({word2domain[word]} 분야) — 상위 {TOPN}개 이웃')
    display(neighbor_table(word, models))
    top_c, top_s = ([w for w, _ in m.wv.most_similar(word, topn=TOPN)] for m in models.values())
    print(f'  상위3 집합 동일: {set(top_c[:3]) == set(top_s[:3])} | 상위3 순서 동일: {top_c[:3] == top_s[:3]}'
          f' | 상위{TOPN} 겹침: {len(set(top_c) & set(top_s))}/{TOPN}')

print('\n■ 분야 대상어 16개(4분야×4단어) 전체 요약 (seed=42, window=2)')
display(pd.DataFrame({name: domain_summary(m) for name, m in models.items()}).T.round(3))

■ 사과 (음식 분야) — 상위 5개 이웃


CBOW            Skip-gram           
     이웃  분야    cos        이웃  분야    cos
순위                                     
1    포도  음식  0.997       바나나  음식  0.998
2   바나나  음식  0.997        딸기  음식  0.997
3    딸기  음식  0.997        포도  음식  0.997
4    주스   -  0.769       만든다   -  0.722
5   강아지  동물  0.650        시장   -  0.697

  상위3 집합 동일: True | 상위3 순서 동일: False | 상위5 겹침: 3/5
■ 노트북 (제품 분야) — 상위 5개 이웃


CBOW            Skip-gram           
     이웃  분야    cos        이웃  분야    cos
순위                                     
1   태블릿  제품  0.999       컴퓨터  제품  0.998
2   휴대폰  제품  0.998       태블릿  제품  0.997
3   컴퓨터  제품  0.998       휴대폰  제품  0.997
4    영화  영화  0.924       드라마  영화  0.677
5   드라마  영화  0.921        공연  영화  0.675

  상위3 집합 동일: True | 상위3 순서 동일: False | 상위5 겹침: 4/5

■ 분야 대상어 16개(4분야×4단어) 전체 요약 (seed=42, window=2)


,같은분야@3,같은분야 cos,다른분야 cos,분리폭,영화–제품 cos
CBOW,1.0,0.998,0.556,0.442,0.917
Skip-gram,1.0,0.998,0.495,0.503,0.671


In [10]:
# [과제] 순위 안정성 — seed만 0~9로 바꿔 10번씩 학습 (나머지 설정은 COMMON과 동일)
stability = []
for name, sg_flag in [('CBOW', 0), ('Skip-gram', 1)]:
    tops = {w: [] for w in COMPARE_WORDS}
    for s in range(10):
        m = Word2Vec(sentences=sentences, sg=sg_flag, **{**COMMON, 'seed': s})
        for w in COMPARE_WORDS:
            tops[w].append([n for n, _ in m.wv.most_similar(w, topn=5)])
    for w in COMPARE_WORDS:
        rank45 = Counter(n for top in tops[w] for n in top[3:])
        stability.append({'model': name, 'word': w,
                          '상위3 집합 종류': len({frozenset(t[:3]) for t in tops[w]}),
                          '상위3 순서 종류': len({tuple(t[:3]) for t in tops[w]}),
                          '4~5위 20자리의 단어 (등장 횟수)': ', '.join(f'{n}({c})' for n, c in rank45.most_common())})
display(pd.DataFrame(stability))

,model,word,상위3 집합 종류,상위3 순서 종류,4~5위 20자리의 단어 (등장 횟수)
0,CBOW,사과,1,5,"주스(6), 토끼(4), 강아지(4), 고양이(3), 개(3)"
1,CBOW,노트북,1,4,"연극(8), 드라마(6), 영화(4), 공연(2)"
2,Skip-gram,사과,1,5,"만든다(9), 시장(6), 아침(5)"
3,Skip-gram,노트북,1,4,"연극(7), 영화(5), 공연(5), 드라마(3)"


#### [해석] 4-1. CBOW와 Skip-gram의 순위 차이

| 관찰 | CBOW | Skip-gram |
|---|---|---|
| 상위 3 | '사과' → 포도·바나나·딸기 / '노트북' → 태블릿·휴대폰·컴퓨터 | 같은 세 단어 (순서만 다름) |
| 상위 3 안정성 (seed 10회) | 집합 1종류, 순서 4–5종류 | 집합 1종류, 순서 4–5종류 |
| '사과' 4–5위 (seed=42) | 주스 0.769, **강아지(동물) 0.650** | 만든다 0.722, 시장 0.697 |
| '사과' 4–5위 (seed 10회, 20자리) | **동물 단어 14자리**(토끼 4·강아지 4·고양이 3·개 3), 주스 6 | 모두 음식 템플릿 단어(만든다 9·시장 6·아침 5) |
| '노트북' 4–5위 (seed=42) | 영화 0.924, 드라마 0.921 | 드라마 0.677, 공연 0.675 |
| '노트북' 4–5위 (seed 10회, 20자리) | 모두 영화 분야(연극 8·드라마 6·영화 4·공연 2) | 모두 영화 분야(연극 7·영화 5·공연 5·드라마 3) |
| 분야 대상어 16개: 다른 분야 평균 cos (seed=42) | 0.556 | 0.495 |
| 영화–제품 평균 cos (seed=42) | 0.917 | 0.671 |

1. **공통점 — 상위 3은 두 모델 모두 같은 분야 단어 (cos ≈ 0.996–0.999).** 같은 분야 4단어는 정확히 같은 5개 템플릿에 들어가 문맥 분포가 완전히 같으므로, 분포 가설대로 거의 같은 벡터가 됩니다. 언어 의미를 학습했다기보다 **말뭉치를 그렇게 만들었기 때문**입니다.
2. **상위 3 안의 순서는 해석하지 않습니다.** 두 모델의 순서가 다르고, seed만 바꿔도 10회 중 4–5가지 순서가 나오며, 1–3위의 cos 차이가 0.003 미만인 사실상 동점입니다.
3. **차이는 4–5위와 점수 수준에서 나타납니다.** CBOW는 다른 분야 단어를 더 가깝게 둡니다. seed=42에서 '사과' → '강아지', '노트북' → 영화 분야 0.92였고, seed 10회 동안 '사과'의 4–5위 20자리 중 14자리가 동물 단어였습니다. Skip-gram은 같은 분야 템플릿 속 단어('만든다', '시장', '아침')만 올라오고, '노트북'의 4–5위가 똑같이 영화 분야여도 cos가 0.68로 훨씬 낮습니다. **순위만 보면 같아 보여도 점수 간격은 다릅니다.** 이 경향은 seed 6개 평균에서도 유지됩니다(4-2 표의 window=2 행: 다른 분야 cos CBOW 0.584±0.028, Skip-gram 0.502±0.008).
4. **가능한 설명 (이 실험만으로 확정할 수는 없음):** 영화·제품 템플릿은 '{w} 의 …', '나는 {w} 를 … 한다' 틀을 공유합니다(4-2의 문맥 분포 표). CBOW는 윈도우 안 문맥 벡터의 **평균**으로 중심 단어를 맞히므로, 같은 윈도우의 다른 단어(예: '화면')가 이미 예측을 해내면 '노트북' 벡터가 분야 고유 정보를 담아야 할 압력이 약해지고 공유된 기능어 문맥 쪽으로 끌립니다. Skip-gram은 '노트북' 벡터 하나로 '화면'·'배터리'·'의'·'를'을 **각각** 예측해야 하므로 내용어 정보가 벡터에 직접 남습니다.
5. **한계:** 템플릿 말뭉치 1개와 seed 10개의 결과이므로 "Skip-gram이 더 좋다"가 아니라 "이 말뭉치·설정에서 Skip-gram이 템플릿의 분야 구조를 더 뚜렷하게 분리했다"까지만 말할 수 있습니다.

### [선택 확장] 4-2. window=1 vs window=3
`window`만 바꾸고 나머지는 `COMMON`과 같게 학습했습니다. (1) 학습 전에 **실습 A의 `context_words`** 로 분야별 문맥 분포를 세어 window에 따라 분야 간 문맥이 얼마나 겹치는지 보고, (2) 두 단어의 순위 변화를 기록한 뒤, (3) seed 6개 평균으로 경향이 우연인지 확인합니다.

gensim은 `window`를 상한으로 두고 실제 윈도우를 1–window 사이에서 무작위로 줄여 쓰므로(`shrink_windows=True`), (1)의 고정 윈도우 통계와 학습 때의 문맥이 정확히 같지는 않습니다. 아래 지표는 학습 말뭉치 안의 진단값이므로 **어느 window가 '최적'이라는 주장은 하지 않습니다**(그러려면 별도 검증 데이터와 평가가 필요).

In [11]:
# [선택 확장] (1) 학습 전 통계 — 분야 대상어가 중심일 때의 문맥 단어 분포를 window별로 비교 (고정 윈도우)
def domain_context_profile(window):
    profile = {d: Counter() for d in groups}
    for tokens, dom in zip(sentences, corpus_df.domain):
        for i, tok in enumerate(tokens):
            if tok in word2domain:
                profile[dom].update(context_words(tokens, i, window=window))   # 실습 A 함수 재사용
    return profile

def counter_cosine(c1, c2):
    keys = sorted(set(c1) | set(c2))
    return cosine_similarity_manual([c1[k] for k in keys], [c2[k] for k in keys])  # 실습 C 함수 재사용

print('말뭉치 최빈 토큰:', Counter(t for sent in sentences for t in sent).most_common(6))
domains = list(groups)
for w in [1, 3]:
    prof = domain_context_profile(w)
    print(f'■ window={w}: 분야 간 문맥 분포 코사인')
    display(pd.DataFrame([[counter_cosine(prof[a], prof[b]) for b in domains] for a in domains],
                         index=domains, columns=domains).round(3))
    for d in ['영화', '제품']:
        print(f'  {d} 대상어의 주요 문맥:', prof[d].most_common(6))

말뭉치 최빈 토큰: [('를', 36), ('의', 28), ('나는', 16), ('한다', 16), ('에서', 12), ('와', 12)]
■ window=1: 분야 간 문맥 분포 코사인


,동물,음식,영화,제품
동물,1.000,0.386,0.373,0.132
음식,0.386,1.000,0.483,0.245
영화,0.373,0.483,1.000,0.889
제품,0.132,0.245,0.889,1.000


  영화 대상어의 주요 문맥: [('의', 12), ('를', 8), ('나는', 4), ('와', 4)]
  제품 대상어의 주요 문맥: [('의', 16), ('나는', 4), ('를', 4), ('새로운', 4)]
■ window=3: 분야 간 문맥 분포 코사인


,동물,음식,영화,제품
동물,1.000,0.418,0.314,0.144
음식,0.418,1.000,0.307,0.156
영화,0.314,0.307,1.000,0.634
제품,0.144,0.156,0.634,1.000


  영화 대상어의 주요 문맥: [('의', 12), ('를', 8), ('나는', 4), ('감상', 4), ('한다', 4), ('배우', 4)]
  제품 대상어의 주요 문맥: [('의', 16), ('이', 8), ('나는', 4), ('를', 4), ('사용', 4), ('한다', 4)]


In [12]:
# [선택 확장] (2) window=1 vs window=3 — 두 단어의 상위 5개 이웃 (표기: 이웃[분야] cos, 분야 대상어가 아니면 [분야] 생략)
def fmt(word, score):
    tag = f'[{word2domain[word]}]' if word in word2domain else ''
    return f'{word}{tag} {score:.3f}'

win_models = {(name, w): Word2Vec(sentences=sentences, sg=sg_flag, **{**COMMON, 'window': w})
              for name, sg_flag in [('CBOW', 0), ('Skip-gram', 1)] for w in [1, 3]}
for word in COMPARE_WORDS:
    print(f'■ {word} ({word2domain[word]} 분야)')
    display(pd.DataFrame({f'{name} w={w}': [fmt(n, s) for n, s in m.wv.most_similar(word, topn=5)]
                          for (name, w), m in win_models.items()},
                         index=pd.RangeIndex(1, 6, name='순위')))

■ 사과 (음식 분야)


,CBOW w=1,CBOW w=3,Skip-gram w=1,Skip-gram w=3
순위,,,,
1,딸기[음식] 0.998,바나나[음식] 0.997,바나나[음식] 0.998,바나나[음식] 0.998
2,포도[음식] 0.998,포도[음식] 0.996,딸기[음식] 0.998,딸기[음식] 0.998
3,바나나[음식] 0.997,딸기[음식] 0.996,포도[음식] 0.997,포도[음식] 0.997
4,주스 0.866,주스 0.737,주스 0.849,만든다 0.734
5,준다 0.802,에 0.719,준다 0.794,산다 0.675


■ 노트북 (제품 분야)


,CBOW w=1,CBOW w=3,Skip-gram w=1,Skip-gram w=3
순위,,,,
1,컴퓨터[제품] 0.998,휴대폰[제품] 0.998,태블릿[제품] 0.998,태블릿[제품] 0.998
2,태블릿[제품] 0.998,태블릿[제품] 0.998,휴대폰[제품] 0.998,컴퓨터[제품] 0.998
3,휴대폰[제품] 0.998,컴퓨터[제품] 0.997,컴퓨터[제품] 0.998,휴대폰[제품] 0.997
4,연극[영화] 0.961,연극[영화] 0.903,공연[영화] 0.915,밝다 0.787
5,공연[영화] 0.957,공연[영화] 0.889,영화[영화] 0.913,품질 0.663


In [13]:
# [선택 확장] (3) seed 6개(42, 0~4) 평균 ± 표준편차 — window=2는 기준선, window 외 설정은 COMMON과 동일
rows = []
for w in [1, 2, 3]:
    for name, sg_flag in [('CBOW', 0), ('Skip-gram', 1)]:
        for s in [SEED, 0, 1, 2, 3, 4]:
            m = Word2Vec(sentences=sentences, sg=sg_flag, **{**COMMON, 'window': w, 'seed': s})
            rows.append({'window': w, 'model': name, **domain_summary(m)})
window_summary = (pd.DataFrame(rows).groupby(['window', 'model'])
                  [['같은분야@3', '다른분야 cos', '분리폭', '영화–제품 cos']].agg(['mean', 'std']).round(3))
display(window_summary)

같은분야@3      다른분야 cos           분리폭        영화–제품 cos       
                   mean  std     mean    std   mean    std      mean    std
window model                                                               
1      CBOW         1.0  0.0    0.751  0.015  0.248  0.015     0.968  0.004
       Skip-gram    1.0  0.0    0.717  0.005  0.281  0.005     0.916  0.009
2      CBOW         1.0  0.0    0.584  0.028  0.414  0.028     0.915  0.005
       Skip-gram    1.0  0.0    0.502  0.008  0.495  0.008     0.683  0.018
3      CBOW         1.0  0.0    0.549  0.025  0.448  0.026     0.898  0.012
       Skip-gram    1.0  0.0    0.433  0.012  0.565  0.012     0.593  0.011

#### [해석] 4-2. window를 바꿨을 때의 순위 변화

| | window=1 | window=3 |
|---|---|---|
| '사과' 4–5위 (seed=42) | CBOW 주스·**준다**, Skip-gram 주스·**준다** | CBOW 주스·에, Skip-gram **만든다·산다** |
| '노트북' 4–5위 (seed=42) | CBOW 연극 0.961·공연 0.957, Skip-gram 공연 0.915·영화 0.913 (모두 영화 분야) | CBOW 연극 0.903·공연 0.889, Skip-gram **밝다 0.787·품질 0.663** (제품 템플릿 단어) |
| 영화–제품 문맥 분포 cos (학습 전 통계) | 0.889 | 0.634 |
| 다른 분야 평균 cos (seed 6개) | CBOW 0.751, Skip-gram 0.717 | CBOW 0.549, Skip-gram 0.433 |
| 영화–제품 평균 cos (seed 6개) | CBOW 0.968, Skip-gram 0.916 | CBOW 0.898, Skip-gram 0.593 |

- **상위 3의 구성은 변하지 않고 순서만 바뀝니다.** 모든 window·모델·seed에서 같은분야@3 = 1.0입니다(같은 분야 4단어의 문맥은 window와 상관없이 동일하므로). 순서는 바뀝니다(예: '사과' CBOW window=1 딸기·포도·바나나 → window=3 바나나·포도·딸기).
- **window=1은 바로 옆 단어만 봅니다(이 말뭉치에서는 대부분 조사).** 영화·제품 대상어의 window=1 문맥에서 가장 많은 것이 바로 뒤 조사 '의'(영화 12회, 제품 16회)라서 두 분야의 문맥 분포 cos가 0.889이고, '노트북'의 4–5위가 영화 분야 단어(cos 0.91–0.96)로 채워집니다. 같은 조사 앞에 오는 명사끼리 가까워지는 **통사적·위치적 유사성**입니다. 반면 '사과'의 5위 '준다'(동물 템플릿 '먹이 를 준다'의 동사)는 품사가 다른데도 '를'과 붙어 나온다는 문맥 하나만 공유해 끌려온 것이므로, 작은 말뭉치의 **국소 공기 효과**로 봐야 합니다.
- **window=3은 같은 문장의 내용어까지 봅니다.** '감상'·'배우' 대 '사용'·'화면'이 문맥에 들어오면서 영화–제품 문맥 cos가 0.634로 내려가고, Skip-gram '노트북'의 4–5위가 '밝다'·'품질'(제품 템플릿 단어)로, '사과'는 '만든다'·'산다'로 바뀝니다 → **주제적 관련성**이 더 반영됩니다. seed 6개 평균으로도 window 1→2→3에서 다른 분야 cos가 계속 줄어듭니다(Skip-gram 0.717→0.502→0.433, CBOW 0.751→0.584→0.549).
- **CBOW는 window를 키워도 영화–제품을 거의 분리하지 못합니다**(0.968→0.898). 4-1에서 본 모델 차이가 window가 커질수록 벌어집니다.
- **주의:** 분리폭이 커졌다고 window=3이 '더 좋은' 설정은 아닙니다. 이 지표는 말뭉치 생성 규칙(분야)을 얼마나 복원했는지일 뿐이며, 품사·문법적 유사성이 필요한 과제라면 작은 window가 나을 수도 있습니다. 최적 window는 목적에 맞는 별도 평가로 정해야 합니다.

### 해석 질문
- 유사어 상위 결과가 기대와 다르다면 무엇을 확인할까요?
- vector_size를 키우면 품질이 반드시 좋아질까요?
- '왕-남자+여자≈여왕'은 언제나 재현되나요?


In [14]:
# [해석 질문 근거] ① 비유 단어가 학습 어휘에 있는가  ② vector_size만 바꿀 때 진단 지표 (Skip-gram, seed 0~4)
analogy_words = ['왕', '남자', '여자', '여왕']
print('비유 단어의 학습 어휘 포함 여부:', {w: w in skipgram.wv.key_to_index for w in analogy_words})

n_tokens = sum(len(s) for s in sentences)
rows, top_examples = [], {}
for size in [2, 8, 32, 128]:
    for s in range(5):
        m = Word2Vec(sentences=sentences, sg=1, **{**COMMON, 'vector_size': size, 'seed': s})
        rows.append({'vector_size': size, **domain_summary(m)})
        if s == 0:
            top_examples[size] = [n for n, _ in m.wv.most_similar('사과', topn=5)]
size_summary = pd.DataFrame(rows).groupby('vector_size')[['같은분야@3', '분리폭']].agg(['mean', 'std']).round(3)
size_summary[('학습 파라미터 수', '입력+출력')] = [2 * len(vocabulary) * size for size in size_summary.index]
display(size_summary)
print(f'학습 토큰 수: {n_tokens}개 (80문장)')
for size, top in top_examples.items():
    print(f"vector_size={size:>3}, seed=0 '사과' 상위5: {top}")

비유 단어의 학습 어휘 포함 여부: {'왕': False, '남자': False, '여자': False, '여왕': False}


같은분야@3           분리폭        학습 파라미터 수
              mean    std   mean    std     입력+출력
vector_size                                      
2            0.612  0.117  0.405  0.038       300
8            1.000  0.000  0.487  0.052      1200
32           1.000  0.000  0.494  0.008      4800
128          1.000  0.000  0.488  0.006     19200

학습 토큰 수: 448개 (80문장)
vector_size=  2, seed=0 '사과' 상위5: ['먹이', '포도', '딸기', '신선한', '에게']
vector_size=  8, seed=0 '사과' 상위5: ['바나나', '포도', '딸기', '아침', '만든다']
vector_size= 32, seed=0 '사과' 상위5: ['포도', '바나나', '딸기', '만든다', '아침']
vector_size=128, seed=0 '사과' 상위5: ['포도', '바나나', '딸기', '만든다', '아침']


### [답변] 해석 질문
**Q1. 유사어 상위 결과가 기대와 다르다면 무엇을 확인할까요?**
1. **단어·토큰화:** 대상 단어가 `key_to_index`에 실제로 있는지(FastText는 미등록어도 조회되므로 특히 주의), 토큰 단위(어절/형태소, 조사 분리 여부)와 표기·정규화가 기대와 맞는지.
2. **말뭉치가 기대를 지지하는지:** 빈도(여기 대상어는 각 5회뿐)와 실제 문맥 분포. `context_words`로 문맥을 세어 보면 4-2처럼 조사 같은 기능어가 문맥을 지배해 다른 분야가 가까워지는 원인을 바로 확인할 수 있습니다.
3. **설정:** `sg`가 의도대로인지(실습 B처럼 기본값이 CBOW), `window`, `vector_size`, `epochs`, `negative`, `min_count`, `sample`(여기서는 0이라 빈번한 조사가 그대로 학습됨).
4. **무작위성:** seed를 바꿔도 같은 결과인지(4-1: 상위 3 집합은 10/10 같지만 순서와 4–5위는 바뀜). 라이브러리 버전과 스레드 수도 기록합니다.
5. **기대 자체:** 코사인은 '비슷한 문맥에 나오는 정도'이지 동의어 여부가 아닙니다. 반의어나 같은 문형을 쓰는 다른 분야 단어도 높게 나올 수 있으므로, 기대가 맞는지 판단하려면 별도의 평가 데이터(유사도 데이터셋, 하위 과제)가 필요합니다.

**Q2. vector_size를 키우면 품질이 반드시 좋아질까요?** — 아닙니다. 위 실험(Skip-gram, seed 0–4)에서 2차원은 같은분야@3이 평균 0.612(표준편차 0.117)로 분야 구분이 불완전했고(seed 0에서는 '사과'의 1위가 '먹이'), 8차원 이상은 모두 1.000이었지만 분리폭은 0.487 → 0.494 → 0.488로 더 좋아지지 않았습니다. 반면 학습 파라미터는 1,200개(8차원)에서 19,200개(128차원)로 늘어나는데 학습 토큰은 448개뿐입니다. 데이터에 비해 차원이 크면 일반적으로 과적합과 계산 낭비의 위험이 커집니다(이 실험에서는 분리폭의 표준편차가 오히려 줄어 불안정해지지는 않았음). 너무 작으면 정보가 손실되고 너무 크면 데이터가 부족해지므로, 적정 차원은 말뭉치 규모와 과제에 맞춰 **검증 데이터로 선택**해야 합니다. (여기 지표도 템플릿 구조를 복원한 정도일 뿐 '품질' 자체는 아닙니다.)

**Q3. '왕-남자+여자≈여왕'은 언제나 재현되나요?** — 아닙니다.
- 이 말뭉치에는 네 단어가 하나도 없어(`key_to_index` 모두 False) Word2Vec으로는 계산 자체가 불가능합니다(KeyError). FastText는 벡터를 '만들어' 주지만 실습 D에서 보듯 의미를 보장하지 않습니다.
- 대규모 말뭉치에서도 성별 같은 관계가 일관된 벡터 차이(평행 이동)로 학습되려면 해당 단어들이 충분히, 비슷한 문맥에서 나와야 하며, 결과는 말뭉치·하이퍼파라미터·seed에 따라 달라집니다.
- 평가 방식에도 좌우됩니다. gensim의 `most_similar(positive=['왕', '여자'], negative=['남자'])`는 입력 단어를 후보에서 빼는데, 빼지 않으면 '왕' 같은 입력 단어가 답으로 나오는 경우가 많다는 비판이 있습니다(Nissim et al., 2020).
- 비유가 맞아도 단어의 의미를 '이해'했다는 증거는 아니며, 성별 고정관념 같은 말뭉치 편향이 그대로 드러날 수도 있습니다.

## 5. 문자 n-gram과 FastText
FastText는 단어와 문자 subword 정보를 활용합니다. 문자 n-gram은 형태소 분석이나 BERT의 WordPiece와 다른 단위입니다. 아래 함수는 Unicode 문자 단위 **개념 예시**이며 gensim 내부의 byte 처리·hash bucket을 재현하지 않습니다.

`<강아지>`를 n=2로 분리하면 `<강`, `강아`, `아지`, `지>`이며 n=3이면 `<강아`, `강아지`, `아지>`입니다. 경계 기호도 포함합니다.

In [15]:
def character_ngrams(word, min_n=2, max_n=3):
    bounded = '<' + word + '>'
    return {n: [bounded[i:i+n] for i in range(len(bounded)-n+1)]
            for n in range(min_n, max_n+1)}
print(character_ngrams('강아지'))
fasttext = FastText(sentences=sentences, sg=1, min_n=2, max_n=4,
                    bucket=5000, **COMMON)


{2: ['<강', '강아', '아지', '지>'], 3: ['<강아', '강아지', '아지>']}


### 실습 D · OOV 확인
`강아지랑`은 이 말뭉치에 없지만 subword 조합으로 FastText 벡터를 요청할 수 있습니다. `key_to_index`는 실제 학습 단어 목록입니다. FastText에서 `word in wv`는 벡터 조회 가능성을 포함할 수 있으므로 **학습 어휘 판정에는 key_to_index**를 사용하세요.

벡터 생성 가능≠의미 정확성 보장입니다. 공유 subword가 적거나 덜 학습되었으면 신뢰할 수 없습니다.

In [16]:
oov_word = '강아지랑'
assert oov_word not in vocabulary
rows = []
for name, model in [('Word2Vec', skipgram), ('FastText', fasttext)]:
    learned = oov_word in model.wv.key_to_index
    try:
        vector = model.wv[oov_word]
        available, norm = True, float(np.linalg.norm(vector))
    except KeyError:
        available, norm = False, np.nan
    rows.append({'model': name, 'learned_word': learned,
                 'vector_available': available, 'vector_norm': norm})
display(pd.DataFrame(rows))
assert not rows[0]['vector_available'] and rows[1]['vector_available']
print('FastText OOV 유사어(정답 아님):', fasttext.wv.most_similar(oov_word, topn=3))


,model,learned_word,vector_available,vector_norm
0,Word2Vec,False,False,NaN
1,FastText,False,True,1.701595


FastText OOV 유사어(정답 아님): [('강아지', 0.9994797110557556), ('개', 0.9980525374412537), ('토끼', 0.9977003335952759)]


### 응답 D
`강아지랑`의 학습어휘 포함 여부와 벡터 조회 가능 여부를 각각 적으세요. 완전히 다른 미등록 단어 하나를 직접 골라 같은 방식으로 확인하고, 의미 품질을 보장할 수 없는 이유를 쓰세요.

In [17]:
# [과제] 실습 D — 직접 고른 '완전히 다른' 미등록 단어: '자동차' (말뭉치에 없는 교통 분야 단어) — 위와 같은 방식으로 확인
from gensim.models.fasttext import ft_ngram_hashes, compute_ngrams_bytes

my_oov = '자동차'
assert my_oov not in vocabulary

def oov_rows(word):
    rows = []
    for name, model in [('Word2Vec', skipgram), ('FastText', fasttext)]:
        learned = word in model.wv.key_to_index          # 학습 어휘 판정은 key_to_index
        try:
            vector = model.wv[word]
            available, norm = True, float(np.linalg.norm(vector))
        except KeyError:
            available, norm = False, np.nan
        rows.append({'word': word, 'model': name, 'learned_word': learned, 'word in wv': word in model.wv,
                     'vector_available': available, 'vector_norm': norm})
    return rows

display(pd.DataFrame(oov_rows(oov_word) + oov_rows(my_oov)))
learned_norm = float(np.linalg.norm(fasttext.wv.vectors, axis=1).mean())
print(f'참고: FastText 학습 어휘 벡터의 평균 norm = {learned_norm:.3f}')

# 진단: OOV 벡터는 무엇으로 만들어졌나? — 학습 n-gram 공유 여부(문자 기준)와 gensim 해시 버킷 적중(실제 계산 기준)
BUCKET = fasttext.wv.bucket
train_ngrams = {g for w in vocabulary for gs in character_ngrams(w, 2, 4).values() for g in gs}
bucket_owner = {}
for w in vocabulary:
    for g, h in zip(compute_ngrams_bytes(w, 2, 4), ft_ngram_hashes(w, 2, 4, BUCKET)):
        bucket_owner.setdefault(h, set()).add(g.decode('utf-8'))

diag = []
for w in [oov_word, my_oov, '비행기']:            # '비행기'는 표면 문자 공유 효과를 보기 위한 보조 예시
    grams = [g for gs in character_ngrams(w, 2, 4).values() for g in gs]
    hashes = ft_ngram_hashes(w, 2, 4, BUCKET)
    diag.append({'word': w, 'n-gram 수': len(grams),
                 '학습 n-gram과 공유(문자 기준)': ', '.join(g for g in grams if g in train_ngrams) or '없음',
                 '학습된 버킷 적중(gensim)': f'{sum(h in bucket_owner for h in hashes)}/{len(hashes)}',
                 'norm / 학습어 평균': round(float(np.linalg.norm(fasttext.wv[w])) / learned_norm, 3),
                 '상위 3 이웃(정답 아님)': ', '.join(f'{n} {s:.3f}' for n, s in fasttext.wv.most_similar(w, topn=3))})
display(pd.DataFrame(diag))

trained = np.zeros(BUCKET, dtype=bool)
trained[list(bucket_owner)] = True
ngram_norms = np.linalg.norm(fasttext.wv.vectors_ngrams, axis=1)
print(f'n-gram 버킷 평균 norm: 학습 n-gram이 쓰는 버킷 {ngram_norms[trained].mean():.2f} vs 한 번도 안 쓰인 버킷(초기 난수) {ngram_norms[~trained].mean():.2f}')
for w in [oov_word, my_oov]:
    for g, h in zip(compute_ngrams_bytes(w, 2, 4), ft_ngram_hashes(w, 2, 4, BUCKET)):
        if h in bucket_owner and g.decode('utf-8') not in train_ngrams:
            print(f"해시 충돌: '{w}'의 n-gram '{g.decode('utf-8')}' → 버킷 {h} ← 학습 n-gram {sorted(bucket_owner[h])}과 같은 버킷")

,word,model,learned_word,word in wv,vector_available,vector_norm
0,강아지랑,Word2Vec,False,False,False,NaN
1,강아지랑,FastText,False,True,True,1.701595
2,자동차,Word2Vec,False,False,False,NaN
3,자동차,FastText,False,True,True,0.318184


참고: FastText 학습 어휘 벡터의 평균 norm = 3.336


,word,n-gram 수,학습 n-gram과 공유(문자 기준),학습된 버킷 적중(gensim),norm / 학습어 평균,상위 3 이웃(정답 아님)
0,강아지랑,12,"<강, 강아, 아지, <강아, 강아지, <강아지",7/12,0.510,"강아지 0.999, 개 0.998, 토끼 0.998"
1,자동차,9,없음,1/9,0.095,"먹는다 0.937, 산다 0.906, 로 0.857"
2,비행기,9,"<비, 기>",2/9,0.270,"연기 0.818, 이야기 0.773, 배우 0.715"


n-gram 버킷 평균 norm: 학습 n-gram이 쓰는 버킷 3.18 vs 한 번도 안 쓰인 버킷(초기 난수) 0.10
해시 충돌: '강아지랑'의 n-gram '아지랑>' → 버킷 203 ← 학습 n-gram ['양이']과 같은 버킷
해시 충돌: '자동차'의 n-gram '동차' → 버킷 3474 ← 학습 n-gram ['먹는다']과 같은 버킷


### [답변] 응답 D
**① `강아지랑` (주어진 OOV)**

| 모델 | 학습 어휘 포함 (`key_to_index`) | 벡터 조회 | 비고 |
|---|---|---|---|
| Word2Vec (Skip-gram) | ✗ False | ✗ 불가 (KeyError) | — |
| FastText | ✗ False | ✓ 가능 (norm 1.702) | `'강아지랑' in fasttext.wv`는 True지만 **학습 단어라는 뜻이 아님** |

- 문자 n-gram 12개 중 6개(`<강, 강아, 아지, <강아, 강아지, <강아지`)가 학습 단어 '강아지'와 겹쳐서 이웃이 '강아지' 0.999, '개' 0.998처럼 그럴듯하게 나옵니다.
- 하지만 조사 '랑'(= '와/과')은 말뭉치에 한 번도 없습니다. '랑'이 들어간 나머지 n-gram 6개 중 5개(`지랑`, `랑>`, `아지랑`, `지랑>`, `강아지랑`)는 학습 n-gram이 하나도 해시되지 않은 초기 난수 버킷이고, 1개(`아지랑>`)는 '고양이'의 `양이`와 같은 버킷에 들어간 **해시 충돌**입니다(그래서 gensim 기준 적중이 6이 아니라 7/12). norm도 학습 단어 평균(3.336)의 0.51배입니다. 결과 벡터는 사실상 '강아지' 조각들의 평균이라 '…와 함께'라는 조사 의미는 담기지 않습니다.

**② 직접 고른 단어 `자동차` (말뭉치와 무관한 교통 분야 단어)**

| 모델 | 학습 어휘 포함 (`key_to_index`) | 벡터 조회 | 비고 |
|---|---|---|---|
| Word2Vec (Skip-gram) | ✗ False | ✗ 불가 (KeyError) | — |
| FastText | ✗ False | ✓ 가능 (norm 0.318, 학습 단어 평균의 0.095배) | 이웃: 먹는다 0.937, 산다 0.906, 로 0.857 |

- 문자 기준으로 학습 n-gram과 겹치는 조각이 **하나도 없는데도** 벡터가 만들어지고, 이웃으로 '먹는다'·'산다' 같은 음식 템플릿 단어가 cos 0.9 이상으로 나옵니다.
- 원인: 9개 n-gram 중 `동차`가 학습 n-gram '먹는다'와 **같은 해시 버킷(3474)** 을 씁니다. 나머지 8개는 한 번도 학습되지 않은 초기 난수 버킷이라 크기가 아주 작으므로(버킷 평균 norm: 학습 n-gram이 쓰는 버킷 3.18 vs 초기 난수 0.10), 평균 벡터의 방향이 충돌한 '먹는다' 조각 하나에 끌려간 것입니다. cos가 높다고 믿을 수 있는 것이 아님을 보여 줍니다.
- 보조 예시 `비행기`: 공유 조각이 `<비`('비교')와 `기>`('딸기'·'연기'·'이야기')뿐이라 이웃이 '연기'·'이야기'·'배우'입니다. 끝 글자가 같다는 **표면 형태** 때문입니다.

**③ 의미 품질을 보장할 수 없는 이유**
1. FastText의 OOV 벡터는 그 단어의 문맥으로 학습된 적이 없고, 문자 n-gram 버킷 벡터를 **평균**한 것뿐입니다.
2. 학습 n-gram이 하나도 해시되지 않은 버킷은 **초기 난수** 그대로입니다(학습 말뭉치에 없던 n-gram은 대부분 이런 버킷에 떨어짐).
3. `bucket=5000`의 해시 공간에서 서로 무관한 n-gram이 같은 버킷을 공유하므로(**해시 충돌**) 엉뚱한 정보가 섞입니다('자동차' ↔ '먹는다', '강아지랑' ↔ '고양이').
4. 문자 조각이 같다고 의미가 같지 않습니다(`기>`로 끝나는 단어들, '사과'[과일]/'사과'[사죄]). 문자 n-gram은 형태소가 아니어서 조사·어미의 기능('랑')을 모릅니다.
5. 학습된 n-gram조차 80문장 템플릿 말뭉치에서 온 것이고, 이웃이 맞는지 검증할 평가 데이터가 없습니다.

→ **벡터 생성 가능 ≠ 의미 정확성.** OOV는 `key_to_index`로 학습 여부를 따로 표시하고, 공유 n-gram 수와 norm을 함께 기록해 신뢰도를 판단해야 합니다.

## 6. 평균 문장 벡터
문장의 단어 벡터들을 평균합니다. Word2Vec에서는 OOV를 제외하고, FastText에서는 조회 가능한 OOV를 포함합니다. **모두 누락되거나 빈 문장이면 0벡터**를 반환하며, 길이0 벡터를 유사도 계산에 무조건 넣지 않습니다. coverage도 같이 기록합니다.

두 문장의 토큰 다중집합이 같으면 평균 벡터가 같습니다. 따라서 주어/목적어 순서로 의미가 달라져도 구분하지 못합니다.

### 실습 E
수집한 벡터를 축0 방향으로 평균하는 부분을 완성하세요.

In [18]:
def mean_sentence_vector(text, keyed_vectors):
    tokens = text.split()
    vectors = []
    for token in tokens:
        try:
            vectors.append(keyed_vectors[token])
        except KeyError:
            pass
    coverage = len(vectors) / len(tokens) if tokens else 0.0
    if not vectors:
        return np.zeros(keyed_vectors.vector_size, dtype=np.float32), coverage
    # TODO E (완료): 아래 기본 0벡터를 vectors의 평균으로 바꾸세요. → 축 0(단어 방향) 평균
    return np.mean(np.stack(vectors), axis=0).astype(np.float32), coverage

sentence_a = '강아지 가 고양이 를 좋아한다'
sentence_b = '고양이 가 강아지 를 좋아한다'
vec_a, coverage_a = mean_sentence_vector(sentence_a, skipgram.wv)
vec_b, coverage_b = mean_sentence_vector(sentence_b, skipgram.wv)
print('coverage:', coverage_a, coverage_b)
print('어순 변경 후 평균 벡터 같음:', np.allclose(vec_a, vec_b))
print('완료 확인(영벡터 아님):', np.linalg.norm(vec_a) > 0)
empty_vec, empty_coverage = mean_sentence_vector('', skipgram.wv)
unknown_vec, unknown_coverage = mean_sentence_vector('미등록단어xyz', skipgram.wv)
print('빈/OOV 문장 coverage:', empty_coverage, unknown_coverage)


coverage: 1.0 1.0
어순 변경 후 평균 벡터 같음: True
완료 확인(영벡터 아님): True
빈/OOV 문장 coverage: 0.0 0.0


In [19]:
# [추가 검증] 실습 E — TODO E가 직접 평균과 같은지, coverage가 '학습 어휘 비율'이 아닌 '조회 가능 비율'인지 확인
manual_mean = np.mean([skipgram.wv[t] for t in sentence_a.split()], axis=0)
print('TODO E 결과 = 단어 벡터 직접 평균:', np.allclose(vec_a, manual_mean))

def learned_ratio(text, keyed_vectors):
    tokens = text.split()
    return sum(t in keyed_vectors.key_to_index for t in tokens) / len(tokens) if tokens else 0.0

rows = []
for text in [sentence_a, '강아지랑 산책 한다', '미등록단어xyz']:
    for name, kv in [('Word2Vec', skipgram.wv), ('FastText', fasttext.wv)]:
        vec, cov = mean_sentence_vector(text, kv)
        rows.append({'문장': text, 'model': name, 'coverage (조회 가능 비율)': round(cov, 3),
                     '학습 어휘 비율 (key_to_index)': round(learned_ratio(text, kv), 3),
                     '평균 벡터 norm': round(float(np.linalg.norm(vec)), 3)})
display(pd.DataFrame(rows))

TODO E 결과 = 단어 벡터 직접 평균: True


,문장,model,coverage (조회 가능 비율),학습 어휘 비율 (key_to_index),평균 벡터 norm
0,강아지 가 고양이 를 좋아한다,Word2Vec,1.000,1.000,2.406
1,강아지 가 고양이 를 좋아한다,FastText,1.000,1.000,2.178
2,강아지랑 산책 한다,Word2Vec,0.667,0.667,3.584
3,강아지랑 산책 한다,FastText,1.000,0.667,2.693
4,미등록단어xyz,Word2Vec,0.000,0.000,0.000
5,미등록단어xyz,FastText,1.000,0.000,0.022


## 7. 정적 임베딩에서 순차 모델로
Word2Vec/FastText에서는 같은 단어 표기가 문맥과 무관하게 같은 벡터를 갖습니다. 예를 들어 '배'의 과일/선박 의미를 표기 하나로 자동 분리하지 않습니다. 평균 pooling은 순서까지 버립니다.

RNN은 `h_t = tanh(W_h @ h_{t-1} + W_x @ x_t + b)`로 이전 상태와 현재 입력을 결합합니다. 아래는 **시드 고정 랜덤 가중치로 순서 영향을 보여주는 계산 데모**입니다. 가중치를 학습하지 않았고 감성 분류기도 아닙니다. 상태가 다르다는 사실은 의미를 정확히 이해했다는 뜻이 아닙니다.

In [20]:
rng = np.random.default_rng(SEED)
DIM = skipgram.wv.vector_size
HIDDEN = 8
W_x = rng.normal(0, 0.15, (HIDDEN, DIM))
W_h = rng.normal(0, 0.15, (HIDDEN, HIDDEN))
bias = np.zeros(HIDDEN)

def rnn_demo(text, keyed_vectors):
    h = np.zeros(HIDDEN)
    states = []
    for token in text.split():
        try:
            x = keyed_vectors[token]
        except KeyError:
            x = np.zeros(DIM)
        h = np.tanh(W_h @ h + W_x @ x + bias)
        states.append(h.copy())
    return h, np.array(states)

h_a, states_a = rnn_demo(sentence_a, skipgram.wv)
h_b, states_b = rnn_demo(sentence_b, skipgram.wv)
print('최종 상태 A:', np.round(h_a, 4))
print('최종 상태 B:', np.round(h_b, 4))
print('순서 변경 후 상태 같음:', np.allclose(h_a, h_b))
print('상태 간 L2 거리:', float(np.linalg.norm(h_a-h_b)))
assert states_a.shape == (len(sentence_a.split()), HIDDEN)
assert not np.allclose(h_a, h_b)


최종 상태 A: [-0.3532  0.2242  0.2043 -0.2527  0.8092  0.0337 -0.5022  0.2194]
최종 상태 B: [-0.3541  0.2256  0.2016 -0.2514  0.8095  0.0333 -0.5023  0.2195]
순서 변경 후 상태 같음: False
상태 간 L2 거리: 0.0035078774442922197


In [21]:
# [과제] 평균 pooling vs 랜덤 RNN — 의미가 '바뀌는' 어순 변경(A↔B) vs 의미가 '유지되는' 어순 변경(A↔C)
sentence_c = '고양이 를 강아지 가 좋아한다'   # 조사(가/를)가 역할을 표시하므로 A와 같은 의미 (한국어 어순 뒤섞기)
pairs = {'A↔B 주어·목적어 교체 (의미 바뀜)': (sentence_a, sentence_b),
         'A↔C 조사 붙은 어순 이동 (의미 유지)': (sentence_a, sentence_c)}
rows = []
for label, (s1, s2) in pairs.items():
    m1, _ = mean_sentence_vector(s1, skipgram.wv)
    m2, _ = mean_sentence_vector(s2, skipgram.wv)
    r1, _ = rnn_demo(s1, skipgram.wv)
    r2, _ = rnn_demo(s2, skipgram.wv)
    rows.append({'비교': label,
                 '평균 pooling L2': float(np.linalg.norm(m1 - m2)),
                 '평균 pooling cos': cosine_similarity_manual(m1, m2),
                 '랜덤 RNN 최종상태 L2': float(np.linalg.norm(r1 - r2)),
                 'RNN L2 / ‖h_A‖': float(np.linalg.norm(r1 - r2) / np.linalg.norm(r1))})
display(pd.DataFrame(rows).set_index('비교').round(4))
print("참고: Skip-gram cos('강아지', '고양이') =", round(float(skipgram.wv.similarity('강아지', '고양이')), 4))

# 특정 랜덤 가중치의 우연인지 확인: 가중치를 시드 0~99로 100세트 새로 뽑아 같은 비교 반복
def rnn_final_state(text, keyed_vectors, W_x, W_h, b):
    h = np.zeros(W_h.shape[0])
    for token in text.split():
        try:
            x = keyed_vectors[token]
        except KeyError:
            x = np.zeros(keyed_vectors.vector_size)
        h = np.tanh(W_h @ h + W_x @ x + b)
    return h

wins, ratios = 0, []
for s in range(100):
    r = np.random.default_rng(s)
    Wx_s, Wh_s = r.normal(0, 0.15, (HIDDEN, DIM)), r.normal(0, 0.15, (HIDDEN, HIDDEN))
    hA, hB, hC = (rnn_final_state(t, skipgram.wv, Wx_s, Wh_s, np.zeros(HIDDEN))
                  for t in (sentence_a, sentence_b, sentence_c))
    d_ab, d_ac = np.linalg.norm(hA - hB), np.linalg.norm(hA - hC)
    wins += d_ac > d_ab
    ratios.append(d_ac / d_ab)
print(f'랜덤 가중치 100세트 중 d(A,C) > d(A,B)인 비율: {wins / 100:.2f} | d(A,C)/d(A,B) 중앙값: {np.median(ratios):.0f}배')

,평균 pooling L2,평균 pooling cos,랜덤 RNN 최종상태 L2,RNN L2 / ‖h_A‖
비교,,,,
A↔B 주어·목적어 교체 (의미 바뀜),0.0,1.0,0.0035,0.0032
A↔C 조사 붙은 어순 이동 (의미 유지),0.0,1.0,0.5385,0.4842


참고: Skip-gram cos('강아지', '고양이') = 0.9991


랜덤 가중치 100세트 중 d(A,C) > d(A,B)인 비율: 1.00 | d(A,C)/d(A,B) 중앙값: 124배


#### [해석] 평균 pooling vs RNN — 학습 여부와 의미 한계 구분

| | 평균 pooling | 랜덤 RNN (`rnn_demo`) |
|---|---|---|
| 학습된 부분 | 단어 벡터(Skip-gram)만 학습됨, pooling 자체는 파라미터 없음 | 입력 단어 벡터는 학습됨, **RNN 가중치 W_x·W_h는 seed 고정 랜덤(학습 안 함)** |
| A↔B (주어·목적어 교체, 의미 바뀜) | 같은 벡터 (L2 ≈ 0, cos 1.0) → **구분 못 함** | L2 0.0035 (‖h_A‖의 0.32%) → **거의 같음** |
| A↔C (조사 붙은 어순 이동, 의미 유지) | 같은 벡터 → '같다'고 나오지만 순서를 버린 결과일 뿐 | L2 0.5385 (‖h_A‖의 48%) → **크게 다름** |

- **평균 pooling:** 덧셈은 순서를 바꿔도 결과가 같으므로(순열 불변) 토큰 다중집합이 같은 A·B·C가 모두 같은 벡터입니다. B를 구분하지 못하는 것이 핵심 한계이고, C를 같다고 본 것은 의미를 이해해서가 아니라 순서를 무시했기 때문입니다.
- **랜덤 RNN의 A↔B 차이가 작은 이유:** Skip-gram에서 cos('강아지', '고양이') = 0.9991로 두 단어 벡터가 거의 같아서, 자리를 바꿔도 입력이 거의 바뀌지 않습니다. RNN이 순서를 처리하더라도 **입력 표현이 두 단어를 구분하지 못하면** '누가 누구를 좋아하는지'를 구분할 수 없습니다(템플릿 말뭉치가 두 단어를 같은 문맥에만 넣은 결과).
- **A↔C 차이가 큰 이유:** 0–3번 위치의 입력이 모두 바뀌기 때문입니다. 의미는 같지만, 학습하지 않은 RNN은 '가/를'이 역할을 표시한다는 규칙을 모르고 **표면 순서가 바뀐 양**에만 반응합니다. 랜덤 가중치를 100세트 새로 뽑아도 100%에서 d(A,C) > d(A,B)였고(중앙값 124배), 의미가 같은 C를 의미가 다른 B보다 훨씬 멀게 표현했습니다.
- **결론:** 상태가 다르다는 것은 'RNN 연산이 순서에 의존한다'는 구조적 성질의 증거일 뿐, 의미를 구분한다는 증거가 아닙니다. 의미 구분을 주장하려면 의미 라벨(예: 행위자 역할, 문장 쌍의 의미 동일 여부)로 RNN을 **학습**하고 분리된 test로 **평가**해야 합니다. 학습된 RNN의 은닉 상태는 문맥에 따라 달라지므로 '배' 같은 다의어를 어느 정도 구분할 수 있지만(ELMo가 이런 방식), 입력 임베딩은 여전히 표기당 하나이고, 단방향 RNN은 뒤 문맥을 보지 못하며, 장기 의존성·기울기 소실 문제도 남습니다.

### RNN의 한계와 10주차 연결
- 순차 처리 때문에 훈련 병렬화가 제한됩니다.
- 긴 문장의 정보 보존과 학습에서 장기 의존성·기울기 소실 문제가 생길 수 있습니다.
- LSTM/GRU는 게이트로 이를 완화하지만 모든 문제를 해결하지는 않습니다.
- 다음 주 Attention/Transformer에서는 토큰 간 관계를 직접 계산하고 위치 정보를 추가합니다.
- Doc2Vec은 문서 벡터를 함께 학습하는 별도 방법이며, 단어 평균이나 완전한 어순 이해와 같지 않습니다.

## 8. 임베딩을 분류에 쓸 때의 데이터 누수
이번 노트북은 **표현 학습·탐색** 실습이며 라벨 분류 성능을 평가하지 않습니다. 임베딩을 분류기에 연결해 평가한다면:
1. 원문을 train/validation/test로 먼저 분할합니다.
2. 직접 학습한 임베딩은 train 텍스트만으로 학습합니다.
3. validation으로 차원/window/분류기 등을 결정합니다.
4. 확정 후 test를 한 번 평가합니다.
5. 외부 사전학습 임베딩은 출처·라이선스·학습 데이터 겹침 위험을 기록합니다.

test 문장을 라벨 없이 임베딩 학습에 사용해도 일반적인 독립 평가에서는 누수입니다. transductive 설정이라면 이를 별도로 명시해야 합니다.

## 9. 형성평가·보고서 과제
### 확인 질문
1. CBOW/Skip-gram의 입력과 예측 대상을 쓰세요.
2. cosine=0의 기하학적 뜻은 무엇인가요?
3. FastText OOV 조회가 의미 품질을 보장하지 않는 이유는?
4. 평균 문장벡터가 어순을 구분하지 못하는 이유는?
5. 랜덤 RNN 상태가 다르면 문장 의미를 이해했다고 볼 수 있나요?
6. 분류 실험에서 test를 임베딩 학습에 넣어도 되나요?

### 과제
- 실습 A/B/C/E TODO를 완성하고, D의 OOV 해석을 작성하세요.
- 단어 두 개에 대해 CBOW/SG 유사어 순위를 비교하고 차이를 설명하세요.
- **선택 확장:** window=1과 window=3을 같은 조건으로 학습하여 순위 변화를 기록하세요. 최적 성능을 주장하려면 별도 평가가 필요합니다.
- 평균 pooling과 RNN의 결과를 비교하되 학습 여부와 의미 한계를 구분하세요.
- 말뭉치의 분야별 템플릿 편향, 작은 규모, 어절/형태소 차이 등 한계를 적으세요.

### 응답란
- 중심/문맥 방향:
- 직접 코사인 계산:
- 유사어 비교와 한계:
- OOV 실험 및 해석:
- 어순·평균·RNN 비교:
- 데이터 누수 방지 계획:

### 제출 체크리스트
- [ ] TODO 완료 후 처음부터 순서대로 실행
- [ ] 실행 결과 저장 / 시드·버전·하이퍼파라미터 기록
- [ ] 교육용 자체작성 말뭉치임을 명시
- [ ] OOV 학습어휘와 조회 가능성 구분
- [ ] 의미 품질·분류 성능을 과장하지 않음

제출 예시: `학번_이름_09주차.ipynb` 및 해석을 적은 README. GitHub/LMS 제출.

### [답변] 확인 질문
1. **CBOW/Skip-gram의 입력과 예측 대상:** CBOW는 window 안의 **주변 단어들**(입력 벡터의 평균)을 입력으로 **중심 단어**를 예측합니다. Skip-gram은 **중심 단어**를 입력으로 **주변 단어 각각**을 예측합니다. 예: '나는 귀여운 [강아지] 와 산책'에서 CBOW는 {나는, 귀여운, 와, 산책} → 강아지를 한 번, Skip-gram은 강아지 → 나는/귀여운/와/산책 4쌍을 예측합니다. negative sampling에서는 정답 쌍 1개와 무작위 대조 단어 `negative`=5개를 구분하는 이진 분류로 학습합니다.
2. **cosine = 0의 기하학적 뜻:** 내적이 0, 즉 두 벡터가 **직교(90°)** 하여 한 벡터 방향으로의 성분이 없다는 뜻입니다. 모든 의미 관계가 없다는 증명은 아니며(고차원에서는 무관한 벡터끼리도 거의 직교하고, 다른 측면의 관계는 있을 수 있음), 0벡터가 끼면 정의되지 않아 nan입니다(실습 C 검증). 서로 다른 단어의 one-hot(희소 표현) 벡터는 항상 cos 0이라 '강아지'와 '개'의 유사성을 담지 못하지만, 32차원 분산 표현에서는 0.9969였습니다.
3. **FastText OOV 조회가 의미 품질을 보장하지 않는 이유:** OOV 벡터는 그 단어의 문맥으로 학습된 것이 아니라 문자 n-gram 버킷 벡터의 평균입니다. 학습되지 않은 n-gram은 초기 난수이고, 해시 충돌로 무관한 n-gram 정보가 섞이며, 문자 형태가 비슷하다고 의미가 비슷하지 않습니다. 실습 D에서 '자동차'의 1위 이웃이 '먹는다'(cos 0.937)였습니다.
4. **평균 문장 벡터가 어순을 구분하지 못하는 이유:** 정적 임베딩은 위치와 상관없이 같은 단어에 같은 벡터를 주고, 평균(합)은 더하는 순서를 바꿔도 값이 같습니다(교환법칙, 순열 불변). 그래서 토큰 다중집합이 같은 A·B·C의 평균 벡터가 모두 같았습니다.
5. **랜덤 RNN 상태가 다르면 문장 의미를 이해했다고 볼 수 있나요?** 볼 수 없습니다. 가중치가 학습되지 않았고, 상태 차이는 순서에 의존하는 연산 구조에서 나올 뿐입니다. 실제로 랜덤 가중치 100세트에서 A로부터의 거리가 의미가 같은 C는 의미가 다른 B의 124배(중앙값)였습니다. 이해를 주장하려면 학습과 분리된 평가가 필요합니다.
6. **분류 실험에서 test를 임베딩 학습에 넣어도 되나요?** 일반적인 독립 평가에서는 안 됩니다. 라벨이 없어도 test 문장의 어휘·공기 정보가 임베딩(어휘 목록, n-gram, 벡터 위치)에 들어가 test 분포를 미리 본 셈이 되므로 성능이 과대 추정될 수 있습니다. 꼭 써야 한다면 transductive 설정임을 명시하고 일반 성능과 구분해 보고합니다.

In [22]:
# [제출 기록] 실행 환경·시드·하이퍼파라미터
import os, platform, scipy
from gensim.models import word2vec as gensim_word2vec

record = {
    'Python': sys.version.split()[0],
    'platform': platform.platform(),
    'numpy / pandas / scipy / gensim': f'{np.__version__} / {pd.__version__} / {scipy.__version__} / {gensim.__version__}',
    'gensim FAST_VERSION (0 이상 = C 최적화 루틴)': gensim_word2vec.FAST_VERSION,
    'PYTHONHASHSEED': os.environ.get('PYTHONHASHSEED', '(미설정)'),
    'SEED': SEED,
    'Word2Vec COMMON': COMMON,
    'CBOW / Skip-gram': 'sg=0 / sg=1 (나머지 COMMON 동일)',
    'FastText': 'sg=1, min_n=2, max_n=4, bucket=5000 + COMMON',
    'RNN 데모': f'HIDDEN={HIDDEN}, W~N(0, 0.15²), bias=0, rng=np.random.default_rng(SEED), 학습 안 함',
    '추가 실험': 'window∈{1,2,3}·seed∈{42,0~4} / vector_size∈{2,8,32,128}·seed 0~4 / 순위 안정성 seed 0~9 / RNN 가중치 seed 0~99',
    '말뭉치': f'교육용 자체 작성 템플릿 {len(sentences)}문장 (4분야×4단어×5템플릿), 고유 토큰 {len(vocabulary)}개, 실제 뉴스/리뷰 아님',
}
for key, value in record.items():
    print(f'{key}: {value}')

Python: 3.13.18
platform: Linux-6.18.44-fc-v310-x86_64-with-glibc2.39
numpy / pandas / scipy / gensim: 2.5.4 / 3.1.1 / 1.18.1 / 4.4.0
gensim FAST_VERSION (0 이상 = C 최적화 루틴): 0
PYTHONHASHSEED: (미설정)
SEED: 42
Word2Vec COMMON: {'vector_size': 32, 'window': 2, 'min_count': 1, 'workers': 1, 'seed': 42, 'epochs': 150, 'negative': 5, 'sample': 0}
CBOW / Skip-gram: sg=0 / sg=1 (나머지 COMMON 동일)
FastText: sg=1, min_n=2, max_n=4, bucket=5000 + COMMON
RNN 데모: HIDDEN=8, W~N(0, 0.15²), bias=0, rng=np.random.default_rng(SEED), 학습 안 함
추가 실험: window∈{1,2,3}·seed∈{42,0~4} / vector_size∈{2,8,32,128}·seed 0~4 / 순위 안정성 seed 0~9 / RNN 가중치 seed 0~99
말뭉치: 교육용 자체 작성 템플릿 80문장 (4분야×4단어×5템플릿), 고유 토큰 75개, 실제 뉴스/리뷰 아님


In [23]:
# [제출 기록] 재현성 점검 — 새 프로세스에서 PYTHONHASHSEED를 바꿔 같은 설정으로 다시 학습한 벡터가 위 Skip-gram과 같은가?
import hashlib, json, subprocess

child_code = '''
import sys, json, hashlib, numpy as np
from gensim.models import Word2Vec
p = json.load(sys.stdin)
m = Word2Vec(sentences=p["sentences"], sg=1, **p["common"])
print(hashlib.sha256(np.ascontiguousarray(m.wv.vectors).tobytes()).hexdigest())
'''
payload = json.dumps({'sentences': sentences, 'common': COMMON})
in_notebook = hashlib.sha256(np.ascontiguousarray(skipgram.wv.vectors).tobytes()).hexdigest()
for hash_seed in ['0', '123']:
    out = subprocess.run([sys.executable, '-c', child_code], input=payload, capture_output=True, text=True,
                         env={**os.environ, 'PYTHONHASHSEED': hash_seed}, check=True).stdout.strip()
    print(f'PYTHONHASHSEED={hash_seed} 새 프로세스: 노트북의 Skip-gram 벡터와 동일 → {out == in_notebook}')
print('노트북 Skip-gram 벡터 SHA-256 앞 16자리:', in_notebook[:16])

PYTHONHASHSEED=0 새 프로세스: 노트북의 Skip-gram 벡터와 동일 → True


PYTHONHASHSEED=123 새 프로세스: 노트북의 Skip-gram 벡터와 동일 → True
노트북 Skip-gram 벡터 SHA-256 앞 16자리: 6855d295511ce44d


### [답변] 응답란
> 수치는 이 노트북의 실행 결과(바로 위 **[제출 기록]** 환경) 기준입니다. 다른 런타임에서 다시 실행하면 소수점 아래 값이나 동점에 가까운 순위는 달라질 수 있습니다. seed 반복은 이 환경에서 확인했으며, 그 결론(상위 3 = 같은 분야, Skip-gram의 분야 분리 > CBOW, window↑ → 분야 분리↑, d(A,C) ≫ d(A,B))은 다른 런타임에서도 유지될 것으로 예상합니다.

- **중심/문맥 방향:** `context_words(example, 2, window=2)` → `['나는', '귀여운', '와', '산책']`(완료 확인 True). CBOW는 이 4개 문맥 → 중심 '강아지'를, Skip-gram은 '강아지' → 문맥 4개(쌍 4개)를 예측합니다. 문장 양 끝(경계)에서는 문맥이 줄고(맨 앞 '나는' → 2개), window=0이면 빈 목록입니다. 이 함수는 고정 윈도우 개념 예시이며, gensim은 학습할 때 실제 윈도우를 1–window에서 무작위로 줄입니다.
- **직접 코사인 계산:** `dot(a, b) / (‖a‖·‖b‖)`를 float64로 계산하고 분모가 0이면 nan을 반환했습니다. cos(강아지, 개)는 직접 계산 0.9969009, gensim 0.9969009로 오차가 1e-6 미만(완료 확인 True)입니다. 어휘 75개의 모든 쌍 2,775개에서 최대 오차는 9.6e-08이었고, 같은 방향 1 / 반대 방향 -1 / 직교 0 / 0벡터 nan도 확인했습니다. 서로 다른 단어의 one-hot(희소, 75차원)은 항상 cos 0이지만 분산 표현(32차원)에서는 0.9969라, 분산 표현이어야 단어 간 유사도를 표현할 수 있습니다.
- **유사어 비교와 한계:** (4-1, 4-2) 두 모델 모두 상위 3은 같은 분야 단어였습니다(cos ≈ 0.996–0.999, seed 10회 모두 같은 집합). 이는 같은 분야 단어가 같은 템플릿을 쓰도록 만든 말뭉치 구성의 결과이며, 상위 3 안의 순서는 동점에 가까워 해석하지 않았습니다. 차이는 4–5위와 점수에서 나타났습니다. CBOW는 다른 분야를 더 가깝게 두었고(seed=42: '노트북' → 영화 분야 0.92, 분야 대상어 16개의 다른 분야 평균 cos 0.556), Skip-gram은 같은 분야 템플릿 단어가 올라오고 영화 분야와는 0.68로 더 멀었습니다(평균 0.495). seed 6개 평균에서도 0.584 vs 0.502로 같은 방향이었습니다. window=1은 조사 '의'를 공유하는 영화·제품을 섞었고, window=3은 내용어까지 보며 분야를 더 분리했습니다(Skip-gram 다른 분야 cos 0.717 → 0.433, seed 6개 평균). 한계: 분야 라벨은 생성 규칙일 뿐이고, 템플릿 말뭉치 하나에 외부 평가가 없으므로 어느 모델·window가 '최적'이라고 주장하지 않습니다.
- **OOV 실험 및 해석:** (실습 D) `강아지랑`과 직접 고른 `자동차` 모두 학습 어휘(`key_to_index`)에는 없고, Word2Vec은 조회 불가, FastText는 조회 가능했습니다. `강아지랑`은 '강아지'와 n-gram 6개를 공유해 이웃이 그럴듯하지만 조사 '랑'의 의미는 없습니다. `자동차`는 공유 n-gram이 없는데도 해시 충돌('동차' ↔ '먹는다') 때문에 1위 이웃이 '먹는다' 0.937로 나왔습니다 → 조회 가능 ≠ 학습 어휘 ≠ 의미 정확성. 평균 문장 벡터의 coverage도 FastText에서는 '조회 가능 비율'이라 '미등록단어xyz'가 1.0이었으므로, 학습 어휘 비율을 따로 기록했습니다(실습 E 검증).
- **어순·평균·RNN 비교:** (7절) 평균 pooling은 토큰 다중집합이 같은 A·B·C가 모두 같은 벡터였습니다. 랜덤 RNN은 A↔B가 L2 0.0035로 거의 같고(cos('강아지', '고양이') = 0.9991), 의미가 같은 A↔C는 0.5385로 컸으며, 랜덤 가중치 100세트 모두 같은 경향이었습니다. 단어 벡터는 학습됐지만 RNN 가중치는 학습되지 않았으므로, 상태 차이는 순서 민감성의 증거일 뿐 의미 이해가 아닙니다.
- **데이터 누수 방지 계획:**
  1. 데이터에서 배우지 않는 고정 규칙 정규화(유니코드 NFC 등)와 중복 제거를 한 뒤, **원문을 먼저** train/validation/test로 나눕니다(예: 60/20/20, 분야 층화, 분할 seed 기록). 이 말뭉치처럼 템플릿으로 만든 데이터는 같은 템플릿이 train과 test에 동시에 들어가면 근접 중복 누수가 되므로 **템플릿(또는 대상 단어) 단위로 묶어서 분할**합니다.
  2. Word2Vec/FastText 학습, 어휘·`min_count`, n-gram 버킷, 빈도 기반 불용어 목록처럼 **데이터에서 배우는 모든 것은 train 텍스트로만** 만듭니다. validation/test의 미등록어는 OOV로 처리하고 coverage와 학습 어휘 비율을 함께 기록합니다.
  3. `vector_size`·`window`·`sg`·`epochs`와 분류기 설정은 **validation으로만** 고르고, 여러 seed의 평균±표준편차로 비교합니다.
  4. 설정을 확정한 뒤(train+validation으로 다시 학습할지도 미리 정함) **test는 한 번만** 평가하고, 그 결과를 보고 다시 조정하지 않습니다.
  5. 외부 사전학습 임베딩은 출처·라이선스·학습 데이터 기간을 기록하고 test 문장과 겹칠 위험을 점검합니다.
  6. 라벨 없는 test 텍스트를 임베딩 학습에 쓰는 경우는 transductive 설정으로 명시하고 일반 성능과 섞어 보고하지 않습니다.

### [과제] 말뭉치·실험의 한계
1. **교육용 자체 작성 데이터:** 템플릿으로 만든 80문장이며 실제 뉴스·리뷰가 아닙니다. 결과를 실제 한국어 단어 의미로 일반화할 수 없습니다.
2. **분야별 템플릿 편향:** 같은 분야 4단어는 정확히 같은 5개 템플릿에 들어가므로 같은분야@3 = 1.0은 설계상 당연한 결과입니다. 분야 사이의 분리도 균일하지 않습니다. 영화·제품은 '{w} 의 …' 틀을 공유해 window=1 문맥 분포 cos가 0.889인 반면 동물·제품은 0.132입니다. 실제 언어에서는 한 단어가 여러 주제·문형에 섞여 나옵니다.
3. **작은 규모:** 토큰 448개, 고유 토큰 75개, 대상어는 각 5회입니다. 상위 3의 집합은 seed 10회 모두 같았지만, 그 안의 순서(10회 중 4–5가지)와 4–5위 단어는 seed에 따라 바뀝니다. 150 epoch 반복은 데이터의 다양성을 늘려 주지 않으며, vector_size=32만 해도 학습 파라미터(4,800개)가 토큰 수의 10배가 넘습니다.
4. **어절/형태소 차이:** 조사를 미리 띄어 쓴 '공백 분할 토큰'일 뿐 형태소 분석 결과가 아니며 기준도 일정하지 않습니다(예: '감상 한다'는 분리, '좋아한다'·'먹는다'는 그대로). 실제 한국어 원문을 공백으로 자르면 '강아지를/강아지와/강아지랑'이 서로 다른 어절 토큰이 되어 희소성과 OOV가 커지고, 형태소 분석기를 쓰면 단위와 결과가 달라집니다. FastText의 문자 n-gram도 형태소가 아닙니다.
5. **기능어 영향:** 최빈 토큰이 '를'(36회)·'의'(28회)일 만큼 템플릿 설계상 조사가 문맥을 지배하는 말뭉치이고(4-2의 window=1 통계), `sample=0`이라 학습에서도 빈번한 토큰을 줄이지 않습니다.
6. **정적 임베딩:** 표기 하나에 벡터 하나이므로 '사과'(과일/사죄), '배' 같은 동형이의어를 구분하지 못합니다.
7. **평가 부재:** 외부 유사도 데이터셋이나 분류 같은 하위 과제로 평가하지 않았습니다. 이웃 순위와 분리폭은 말뭉치 구조를 진단한 값일 뿐 의미 품질이나 분류 성능이 아니며, RNN은 학습하지 않은 개념 데모입니다.
8. **재현성 범위:** 이 환경에서는 PYTHONHASHSEED를 바꾼 새 프로세스에서도 같은 벡터가 나왔지만, 다른 라이브러리 버전·BLAS·CPU에서는 수치가 달라질 수 있습니다.

### [답변] 제출 체크리스트 점검
- [x] **TODO 완료 후 처음부터 순서대로 실행** — 커널을 새로 시작해 전체 실행. 실행 번호가 1부터 끊김 없이 이어지고 오류가 없으며, 완료 확인 A·B·C·E가 모두 True입니다.
- [x] **실행 결과 저장 / 시드·버전·하이퍼파라미터 기록** — 모든 출력 저장. **[제출 기록]** 셀에 Python·numpy·pandas·scipy·gensim 버전, SEED=42, `COMMON`, FastText·RNN 설정, 추가 실험의 seed 목록과 재현성 점검 결과를 남겼고 README에도 같은 내용을 적었습니다.
- [x] **교육용 자체작성 말뭉치임을 명시** — 맨 위 [제출 안내], 한계 1번, README.
- [x] **OOV 학습어휘와 조회 가능성 구분** — `key_to_index`(학습 어휘)와 `in wv`/벡터 조회(가능성)를 따로 표시(실습 D), coverage와 학습 어휘 비율 구분(실습 E 검증).
- [x] **의미 품질·분류 성능을 과장하지 않음** — 분야 라벨은 생성 규칙이고 지표는 진단값임을 명시, 최적 window·모델을 주장하지 않음, 분류 성능은 평가하지 않음, RNN은 학습하지 않은 데모로 해석.

## 참고 자료와 원자료 보완
- 제공 「Part3 딥러닝 9–12주차 강의안」 9주차(5–9쪽)
- 제공 「자연언어처리 강의계획서」 9주차
- [gensim Word2Vec 문서](https://radimrehurek.com/gensim/models/word2vec.html)
- [gensim FastText 문서](https://radimrehurek.com/gensim/models/fasttext.html)

문자 n-gram 경계 예시와 RNN의 h_{t-1} 표기를 명확히 했습니다. 원자료의 소량 말뭉치만으로 의미 품질을 보장하지 않으며, 형태소 분석은 선택 확장으로 구분합니다. 학습 결과와 랜덤 RNN 개념 데모를 구별합니다.

### [추가] 참고 문헌 (해석 작성에 사용)
- Mikolov et al. (2013). *Efficient Estimation of Word Representations in Vector Space.* arXiv:1301.3781 — CBOW·Skip-gram
- Mikolov et al. (2013). *Distributed Representations of Words and Phrases and their Compositionality.* NIPS 2013 — negative sampling·subsampling
- Bojanowski et al. (2017). *Enriching Word Vectors with Subword Information.* TACL 5 — FastText 문자 n-gram
- Nissim, van Noord & van der Goot (2020). *Fair Is Better than Sensational: Man Is to Doctor as Woman Is to Doctor.* Computational Linguistics 46(2) — 비유 평가에서 입력 단어를 후보에서 빼는 문제

### [제출 기록] 외부 AI 사용 및 검증 기록

- 사용 도구: ChatGPT, Claude
- 사용 범위:
  - 실습 A·B·C·E TODO 코드와 직접 추가한 코드 셀([과제]·[선택 확장]·[추가 검증]·[해석 질문 근거]·[제출 기록]): 먼저 직접 작성한 뒤 AI에게 검토를 받았고, 오류가 있을 때 해결 방법을 물어 고쳤습니다.
  - [해석]·[답변] 셀과 한계 정리: 먼저 스스로 생각한 뒤 AI에게 제 생각과 다른 관점을 물어 공부했고, 답변은 직접 쓴 다음 AI로 문장을 다듬었습니다.
  - 외부 자료: 해석에 참고한 강의자료·gensim 문서·논문은 위의 '참고 자료와 원자료 보완'과 '[추가] 참고 문헌' 셀에 적었습니다.
- 검증 과정:
  - AI가 고쳐 준 코드는 다시 실행해 오류가 사라지고 출력이 나오는지 확인했습니다. 제출본은 커널을 새로 시작해 위에서부터 한 번에 실행한 결과이며, 실행 번호 1~23이 끊김 없이 이어지고 오류가 없습니다.
  - [추가 검증]·[제출 기록] 셀로 직접 계산과 라이브러리 결과를 대조했습니다. 예: 실습 C 직접 계산 코사인과 gensim의 최대 절대오차 9.6e-08(어휘 쌍 2,775개), 실습 E 결과와 단어 벡터 직접 평균의 일치(True), PYTHONHASHSEED를 바꾼 새 프로세스에서 다시 학습한 벡터와의 일치(True)
  - AI로 다듬은 답변은 제가 쓴 뜻과 실행 출력에 맞는지 대조했습니다. 예: 응답란의 cos(강아지, 개) 0.9969, 랜덤 RNN 최종 상태 거리 A↔B 0.0035·A↔C 0.5385
  - AI가 알려 준 다른 관점과 설명은 강의자료·gensim 문서·참고 문헌과 비교했습니다.